# RAG-Based Intelligent Document QA System
**Mistral-7B-Instruct + FAISS Dense + BM25 Sparse + Reciprocal Rank Fusion**

Author: Yuvraj Chulpar | NIT Kurukshetra | [GitHub](https://github.com/chulparyuvraj/rag-document-qa)

---
### Pipeline Overview
1. PDF Ingestion → Chunking (512 tokens, 64 overlap)
2. FAISS dense index (sentence-transformers/all-MiniLM-L6-v2)
3. BM25 sparse index (rank_bm25)
4. Hybrid retrieval with Reciprocal Rank Fusion (RRF)
5. Mistral-7B-Instruct-v0.2 in 4-bit QLoRA for answer generation

> ⚠️ **Run Cell 1 first, then restart the kernel, then run all remaining cells in order.**


## Cell 1 — Install Dependencies
> ⚠️ After this cell finishes, **restart the kernel**, then run from Cell 2 onwards.

In [1]:
# Fix numpy FIRST — must be upgraded before any other package
import subprocess
subprocess.run('pip install -q "numpy>=1.26.0" --upgrade', shell=True)

# Install all required packages
!pip install -q \
    pymupdf loguru tqdm rank_bm25 \
    langchain-core langchain-community langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers faiss-cpu \
    transformers peft trl bitsandbytes accelerate datasets \
    rouge-score python-dotenv huggingface_hub

print("✅ All packages installed — now RESTART THE KERNEL then run Cell 2 onwards")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 87.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.


✅ All packages installed — now RESTART THE KERNEL then run Cell 2 onwards


In [5]:
!pip install -q pymupdf loguru tqdm rank_bm25 \
    langchain-core langchain-community langchain-huggingface \
    langchain-text-splitters sentence-transformers faiss-cpu \
    transformers peft trl bitsandbytes accelerate datasets \
    rouge-score python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 74.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.2/504.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is no

## Cell 2 — Project Setup

In [1]:
import sys, os, shutil, subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr and 'error' in result.stderr.lower(): print("STDERR:", result.stderr[:300])

# Verify numpy is correct version
import numpy as np
print(f"numpy version: {np.__version__}")
assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (1, 26), \
    "numpy too old — restart kernel and rerun from Cell 1"
print("✅ numpy OK")

# Copy project to writable working directory
BASE = '/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa'
WORK = '/kaggle/working/rag-document-qa'

if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(BASE, WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(f"✅ Working directory: {os.getcwd()}")


numpy version: 2.0.2
✅ numpy OK
✅ Working directory: /kaggle/working/rag-document-qa


## Cell 3 — Fix Broken Files

In [2]:
# Remove the brace-expanded broken __init__.py
broken = os.path.join(WORK, "src/{ingestion,embeddings,retrieval,generation,finetuning,api}/__init__.py")
if os.path.exists(broken):
    os.remove(broken)
    print("Removed broken __init__.py")

# Ensure all __init__.py exist
for mod in ['src','src/ingestion','src/retrieval','src/pipeline','src/finetuning','src/api']:
    init = f'{WORK}/{mod}/__init__.py'
    os.makedirs(os.path.dirname(init), exist_ok=True)
    if not os.path.exists(init):
        open(init, 'w').close()
        print(f"Created: {mod}/__init__.py")

# Patch all broken LangChain imports across src/
fixes = {
    'from langchain.text_splitter import RecursiveCharacterTextSplitter':
        'from langchain_text_splitters import RecursiveCharacterTextSplitter',
    'from langchain.schema import Document':
        'from langchain_core.documents import Document',
    'from langchain.schema.retriever import BaseRetriever':
        'from langchain_core.retrievers import BaseRetriever',
    'from langchain.callbacks.manager import CallbackManagerForRetrieverRun':
        'from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun',
    'from langchain.prompts import PromptTemplate':
        'from langchain_core.prompts import PromptTemplate',
    'from langchain_community.llms import HuggingFacePipeline':
        'from langchain_huggingface import HuggingFacePipeline',
    'from langchain_community.chains import RetrievalQA': '',
    'from langchain.chains import RetrievalQA': '',
}

for root, dirs, files in os.walk('src'):
    for fname in files:
        if not fname.endswith('.py'): continue
        fpath = os.path.join(root, fname)
        with open(fpath, 'r') as f: content = f.read()
        new_content = content
        for old, new in fixes.items():
            new_content = new_content.replace(old, new)
        if new_content != content:
            with open(fpath, 'w') as f: f.write(new_content)
            print(f"✅ Patched: {fpath}")

print("✅ All imports fixed")


Removed broken __init__.py
✅ Patched: src/retrieval/bm25_retriever.py
✅ Patched: src/retrieval/hybrid_retriever.py
✅ Patched: src/retrieval/vector_store.py
✅ Patched: src/ingestion/chunker.py
✅ Patched: src/pipeline/prompt_templates.py
✅ Patched: src/pipeline/rag_chain.py
✅ All imports fixed


## Cell 4 — Rewrite Retriever Files (LangChain v0.2 Compatible)

In [3]:
# Rewrite bm25_retriever.py
bm25_code = '''
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
from rank_bm25 import BM25Okapi
from loguru import logger
import re


class BM25Retriever(BaseRetriever):
    documents: List[Document]
    bm25: object
    k: int = 5

    class Config:
        arbitrary_types_allowed = True

    @classmethod
    def from_documents(cls, documents: List[Document], k: int = 5) -> "BM25Retriever":
        tokenized_corpus = [cls._tokenize(doc.page_content) for doc in documents]
        bm25 = BM25Okapi(tokenized_corpus)
        logger.info(f"BM25 index built on {len(documents)} documents")
        return cls(documents=documents, bm25=bm25, k=k)

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_k_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:self.k]
        results = []
        for idx in top_k_indices:
            doc = self.documents[idx]
            doc.metadata["bm25_score"] = float(scores[idx])
            results.append(doc)
        return results

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return re.findall(r"[a-z0-9]+", text.lower())
'''

with open('src/retrieval/bm25_retriever.py', 'w') as f:
    f.write(bm25_code)
print("✅ bm25_retriever.py rewritten")

# Rewrite hybrid_retriever.py
hybrid_code = '''
from typing import List, Dict
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
from loguru import logger

from .vector_store import FAISSVectorStore
from .bm25_retriever import BM25Retriever


class HybridRetriever(BaseRetriever):
    faiss_store: FAISSVectorStore
    bm25_retriever: BM25Retriever
    k: int = 5
    rrf_k: int = 60
    dense_weight: float = 0.6
    sparse_weight: float = 0.4

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        fetch_k = self.k * 3
        dense_docs  = self.faiss_store.similarity_search(query, k=fetch_k)
        sparse_docs = self.bm25_retriever._get_relevant_documents(query)
        fused = self._rrf(dense_docs, sparse_docs)
        logger.debug(f"Hybrid: dense={len(dense_docs)}, sparse={len(sparse_docs)}, returning top {self.k}")
        return fused[:self.k]

    def _rrf(self, dense_docs: List[Document], sparse_docs: List[Document]) -> List[Document]:
        scores: Dict[str, float] = {}
        doc_map: Dict[str, Document] = {}
        for rank, doc in enumerate(dense_docs):
            key = doc.page_content
            scores[key] = scores.get(key, 0) + self.dense_weight * (1.0 / (self.rrf_k + rank + 1))
            doc_map[key] = doc
        for rank, doc in enumerate(sparse_docs):
            key = doc.page_content
            scores[key] = scores.get(key, 0) + self.sparse_weight * (1.0 / (self.rrf_k + rank + 1))
            doc_map[key] = doc
        sorted_keys = sorted(scores, key=scores.__getitem__, reverse=True)
        results = []
        for key in sorted_keys:
            doc = doc_map[key]
            doc.metadata["rrf_score"] = round(scores[key], 6)
            results.append(doc)
        return results
'''

with open('src/retrieval/hybrid_retriever.py', 'w') as f:
    f.write(hybrid_code)
print("✅ hybrid_retriever.py rewritten")

# Rewrite rag_chain.py
rag_chain_code = '''
import os
from typing import Optional, List
from pathlib import Path

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_huggingface import HuggingFacePipeline
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig,
)
from peft import PeftModel
import torch
from loguru import logger

from .prompt_templates import RAG_PROMPT
from ..ingestion.pdf_loader import PDFLoader
from ..ingestion.chunker import ResearchPaperChunker
from ..retrieval.embeddings import get_embedding_model
from ..retrieval.vector_store import FAISSVectorStore
from ..retrieval.bm25_retriever import BM25Retriever
from ..retrieval.hybrid_retriever import HybridRetriever


class RAGPipeline:
    def __init__(
        self,
        embedding_model_name: str = None,
        llm_model_name: str = None,
        adapter_path: Optional[str] = None,
        index_path: str = "indexes/faiss",
        device: str = "auto",
        top_k: int = 5,
    ):
        self.embedding_model_name = embedding_model_name or os.getenv(
            "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
        )
        self.llm_model_name = llm_model_name or os.getenv(
            "BASE_LLM", "mistralai/Mistral-7B-Instruct-v0.2"
        )
        self.adapter_path = adapter_path or os.getenv("FINETUNED_ADAPTER_PATH")
        self.index_path = index_path
        self.device = device
        self.top_k = top_k
        self._embeddings       = None
        self._faiss_store      = None
        self._bm25             = None
        self._hybrid_retriever = None
        self._llm_pipeline     = None
        self._chunks: List[Document] = []

    def build_index(self, pdf_dir="data/papers", save_index=True, force_rebuild=False):
        self._embeddings  = get_embedding_model(self.embedding_model_name)
        self._faiss_store = FAISSVectorStore(self._embeddings)

        if not force_rebuild and Path(self.index_path).exists():
            logger.info("Loading existing FAISS index from disk...")
            self._faiss_store.load(self.index_path)
            self._chunks = list(self._faiss_store._index.docstore._dict.values())
        else:
            logger.info("Building new index from PDFs...")
            loader  = PDFLoader(source_dir=pdf_dir)
            pages   = loader.load_all()
            chunker = ResearchPaperChunker(chunk_size=512, chunk_overlap=64)
            self._chunks = chunker.chunk(pages)
            self._faiss_store.build(self._chunks)
            if save_index:
                self._faiss_store.save(self.index_path)

        self._bm25             = BM25Retriever.from_documents(self._chunks, k=self.top_k)
        self._hybrid_retriever = HybridRetriever(
            faiss_store=self._faiss_store,
            bm25_retriever=self._bm25,
            k=self.top_k,
        )
        self._llm_pipeline = self._load_llm()
        logger.info("RAG pipeline ready!")

    def query(self, question: str) -> dict:
        if self._llm_pipeline is None:
            raise RuntimeError("Call build_index() first.")
        logger.info(f"Query: {question}")
        source_docs  = self._hybrid_retriever._get_relevant_documents(question)
        context      = "\\n\\n".join([doc.page_content for doc in source_docs])
        prompt_text  = RAG_PROMPT.format(context=context, question=question)
        response     = self._llm_pipeline.pipeline(
            prompt_text, max_new_tokens=512, temperature=0.1,
            do_sample=True, repetition_penalty=1.15,
        )
        answer = response[0]["generated_text"][len(prompt_text):].strip()
        return {"result": answer, "source_documents": source_docs}

    def get_context(self, question: str) -> List[Document]:
        return self._hybrid_retriever._get_relevant_documents(question)

    def _load_llm(self) -> HuggingFacePipeline:
        logger.info(f"Loading LLM: {self.llm_model_name}")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(self.llm_model_name)
        tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            self.llm_model_name,
            quantization_config=bnb_config,
            device_map=self.device,
            trust_remote_code=True,
        )
        if self.adapter_path and Path(self.adapter_path).exists():
            logger.info(f"Loading QLoRA adapter from {self.adapter_path}")
            model = PeftModel.from_pretrained(model, self.adapter_path)
            model = model.merge_and_unload()
        pipe = pipeline(
            "text-generation", model=model, tokenizer=tokenizer,
            max_new_tokens=512, temperature=0.1, do_sample=True,
            repetition_penalty=1.15,
        )
        return HuggingFacePipeline(pipeline=pipe)
'''

with open('src/pipeline/rag_chain.py', 'w') as f:
    f.write(rag_chain_code)
print("✅ rag_chain.py rewritten")


✅ bm25_retriever.py rewritten
✅ hybrid_retriever.py rewritten
✅ rag_chain.py rewritten


## Cell 5 — Verify All Imports

In [6]:
from src.ingestion.pdf_loader import PDFLoader
from src.ingestion.chunker import ResearchPaperChunker
from src.retrieval.embeddings import get_embedding_model
from src.retrieval.vector_store import FAISSVectorStore
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_retriever import HybridRetriever
from src.pipeline.rag_chain import RAGPipeline
from src.pipeline.prompt_templates import RAG_PROMPT

print("✅ All imports successful!")


✅ All imports successful!


## Cell 6 — HuggingFace Login
> Replace `hf_your_token_here` with your real token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import login


HF_TOKEN = 'hf_add_your_token_here'

login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace")


✅ Logged in to HuggingFace


## Cell 7 — Download Research Papers
Adds arXiv papers to the index. You can add more IDs to the list.

In [8]:
import urllib.request

os.makedirs('data/papers', exist_ok=True)

# Add any arXiv paper IDs you want to index
arxiv_ids = [
    '2512.20363',   # Clust-PSI-PFL (your FL paper)
    '1706.03762',   # Attention Is All You Need
]

for arxiv_id in arxiv_ids:
    out_path = f'data/papers/{arxiv_id}.pdf'
    if not os.path.exists(out_path):
        url = f'https://arxiv.org/pdf/{arxiv_id}.pdf'
        urllib.request.urlretrieve(url, out_path)
        print(f'✅ Downloaded: {arxiv_id}.pdf')
    else:
        print(f'⏭ Already exists: {arxiv_id}.pdf')


✅ Downloaded: 2512.20363.pdf
✅ Downloaded: 1706.03762.pdf


## Cell 8 — Build FAISS + BM25 Index
This embeds all PDF chunks using GPU-accelerated sentence-transformers.
Expected time: ~2 minutes on T4.

In [9]:
# Load & chunk PDFs
loader  = PDFLoader(source_dir='data/papers')
pages   = loader.load_all()
print(f'Pages extracted: {len(pages)}')

chunker = ResearchPaperChunker(chunk_size=512, chunk_overlap=64)
chunks  = chunker.chunk(pages)
print(f'Chunks created:  {len(chunks)}')

# Build FAISS dense index
embedding_model = get_embedding_model(device='cuda')
faiss_store     = FAISSVectorStore(embedding_model)
faiss_store.build(chunks)
faiss_store.save('indexes/faiss')
print("✅ FAISS index saved")

# Build BM25 sparse index + Hybrid retriever
bm25   = BM25Retriever.from_documents(chunks, k=5)
hybrid = HybridRetriever(faiss_store=faiss_store, bm25_retriever=bm25, k=5)
print("✅ BM25 + Hybrid retriever ready")


2026-03-21 15:26:41.788 | INFO     | src.ingestion.pdf_loader:_load_from_directory:69 - Found 2 PDFs in data/papers
Loading PDFs: 100%|██████████| 2/2 [00:00<00:00,  7.40it/s]
2026-03-21 15:26:42.062 | INFO     | src.ingestion.pdf_loader:_load_from_directory:73 - Extracted 29 pages total
2026-03-21 15:26:42.068 | INFO     | src.ingestion.chunker:chunk:64 - Chunked 29 pages → 250 chunks (size=512, overlap=64)
2026-03-21 15:26:42.069 | INFO     | src.retrieval.embeddings:get_embedding_model:35 - Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 on cuda


Pages extracted: 29
Chunks created:  250


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-03-21 15:26:46.390 | INFO     | src.retrieval.embeddings:get_embedding_model:46 - Embedding model loaded successfully
2026-03-21 15:26:46.391 | INFO     | src.retrieval.vector_store:build:41 - Building FAISS index from 250 chunks...
2026-03-21 15:26:47.506 | INFO     | src.retrieval.vector_store:build:52 - FAISS index built successfully
2026-03-21 15:26:47.509 | INFO     | src.retrieval.vector_store:save:67 - FAISS index saved to indexes/faiss
2026-03-21 15:26:47.521 | INFO     | src.retrieval.bm25_retriever:from_documents:23 - BM25 index built on 250 documents


✅ FAISS index saved
✅ BM25 + Hybrid retriever ready


## Cell 9 — Test Hybrid Retrieval (No LLM Needed)
Verify retrieval is working before loading the 14GB model.

In [ ]:
test_query = "What is the PSI clustering method in federated learning?"
docs = hybrid._get_relevant_documents(test_query)

print(f'✅ Retrieved {len(docs)} docs for: "{test_query}"')
print()
for i, d in enumerate(docs):
    print(f"[{i+1}] {d.metadata['source']} — page {d.metadata['page']} | RRF: {d.metadata.get('rrf_score', 'N/A')}")
    print(f"    {d.page_content[:120].strip()}...")
    print()


## Cell 10 — Load Mistral-7B-Instruct (4-bit QLoRA)
> ⏳ Takes **5–8 minutes** on T4. Do not interrupt.
> Requires HuggingFace login (Cell 6) and license agreement on the model page.

In [ ]:
rag = RAGPipeline(index_path='indexes/faiss', top_k=5)

# Reuse already-built index objects — skip rebuilding
rag._embeddings        = embedding_model
rag._faiss_store       = faiss_store
rag._chunks            = chunks
rag._bm25              = bm25
rag._hybrid_retriever  = hybrid

print("Loading Mistral-7B-Instruct-v0.2 in 4-bit... (~5-8 mins on T4)")
rag._llm_pipeline = rag._load_llm()
print("✅ Mistral-7B loaded and ready!")


## Cell 11 — Define ask() Helper Function

In [ ]:
def ask(question, retriever=hybrid, llm=None):
    """Query the RAG pipeline directly."""
    if llm is None:
        llm = rag._llm_pipeline

    # Retrieve relevant chunks
    source_docs = retriever._get_relevant_documents(question)

    # Build context + format prompt
    context     = "\n\n".join([doc.page_content for doc in source_docs])
    prompt_text = RAG_PROMPT.format(context=context, question=question)

    # Generate answer
    response = llm.pipeline(
        prompt_text,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        repetition_penalty=1.15,
    )
    answer = response[0]["generated_text"][len(prompt_text):].strip()
    return {"result": answer, "source_documents": source_docs}

print("✅ ask() function ready")


## Cell 12 — Query Your FL Paper (Clust-PSI-PFL)
Testing domain-specific QA on your research paper.

In [ ]:
questions = [
    "What is the cluster federated Learning?",
    "What datasets were used for evaluation?",
    "What are the main results of this paper?",
    "What is Population Stability Index and how is it used?",
    "What are the baseline methods compared in this paper?",
]

for q in questions:
    result = ask(q)
    print(f"Q: {q}")
    print(f"A: {result['result']}")
    sources = [f"{d.metadata['source']} p.{d.metadata['page']}" for d in result['source_documents']]
    print(f"Sources: {sources}")
    print("-" * 70)


## Cell 13 — Generate QA Pairs for Fine-tuning
Auto-generates training data from your FL paper chunks using Mistral.
Used in Notebook 02 for QLoRA fine-tuning.

In [ ]:
import json, re
from tqdm import tqdm

def generate_qa_from_chunk(chunk_text, llm_pipeline):
    prompt = f"""<s>[INST] Read the following text from a research paper on Federated Learning \
and generate exactly ONE question and answer pair.

Text:
{chunk_text[:600]}

Respond in this exact format:
QUESTION: <your question here>
ANSWER: <your answer here>

Rules:
- Question must be specific and answerable from the text
- Answer must be grounded in the text only
- Do not add any other text [/INST]"""

    response = llm_pipeline.pipeline(
        prompt, max_new_tokens=200, temperature=0.3,
        do_sample=True, repetition_penalty=1.1,
    )
    generated = response[0]["generated_text"][len(prompt):].strip()
    q_match = re.search(r'QUESTION:\s*(.+?)(?=ANSWER:|$)', generated, re.DOTALL)
    a_match = re.search(r'ANSWER:\s*(.+?)$', generated, re.DOTALL)
    if q_match and a_match:
        return {
            "context":  chunk_text[:600],
            "question": q_match.group(1).strip(),
            "answer":   a_match.group(1).strip()
        }
    return None

# Use only FL paper chunks for domain-specific fine-tuning
fl_chunks = [c for c in chunks if '2512.20363' in c.metadata.get('source', '')]
print(f"FL paper chunks: {len(fl_chunks)}")

# Sample every 2nd chunk for diversity (max 60 → ~50 valid pairs)
selected = fl_chunks[::2][:60]
print(f"Generating QA pairs from {len(selected)} chunks...")

qa_pairs = []
for chunk in tqdm(selected):
    pair = generate_qa_from_chunk(chunk.page_content, rag._llm_pipeline)
    if pair:
        qa_pairs.append(pair)

print(f"\n✅ Generated {len(qa_pairs)} QA pairs")
if qa_pairs:
    print(f"\nSample:")
    print(f"Q: {qa_pairs[0]['question']}")
    print(f"A: {qa_pairs[0]['answer'][:200]}...")

# Save to disk
os.makedirs('data', exist_ok=True)
with open('data/fl_qa_pairs.json', 'w') as f:
    json.dump(qa_pairs, f, indent=2)
print(f"\n✅ Saved to data/fl_qa_pairs.json")


In [ ]:
import json

with open('data/fl_qa_pairs.json', 'r') as f:
    qa_pairs = json.load(f)

print(f"Total QA pairs: {len(qa_pairs)}\n")
print("=" * 70)

# Show first 5 pairs
for i, pair in enumerate(qa_pairs[:5]):
    print(f"[{i+1}] Q: {pair['question']}")
    print(f"     A: {pair['answer'][:150]}...")
    print()

# Phase 2: QLoRA Fine-tuning:

# Cell 14

In [11]:
# import json
# from src.finetuning.dataset_prep import prepare_dataset

# # Load generated QA pairs
# with open('data/fl_qa_pairs.json', 'r') as f:
#     qa_pairs = json.load(f)

# # Format into train/val split (90/10)
# paths = prepare_dataset(qa_pairs, output_path='data/finetune_dataset.jsonl')
# print(f"Train: {paths['train']}")
# print(f"Val:   {paths['val']}")

In [12]:
import json, os

qa_pairs = [
    {"context": "Federated Learning (FL) enables collaborative model training across decentralized devices without sharing raw data, preserving privacy.", "question": "What is Federated Learning?", "answer": "Federated Learning is a decentralized machine learning approach that enables collaborative model training across multiple devices without sharing raw data, thereby preserving data privacy."},
    {"context": "Non-IID data distribution is a key challenge in federated learning where clients have heterogeneous data distributions causing biased global models.", "question": "What is the Non-IID problem in federated learning?", "answer": "Non-IID refers to non-independent and identically distributed data, where different clients hold data with different statistical distributions, causing biased global model updates in federated learning."},
    {"context": "Clust-PSI-PFL uses Population Stability Index (PSI) to measure distributional shift between clients and cluster them into homogeneous groups.", "question": "How does Clust-PSI-PFL use PSI?", "answer": "Clust-PSI-PFL uses the Population Stability Index (PSI) to quantify distributional differences between federated clients and cluster them into homogeneous groups for personalized federated learning."},
    {"context": "PSI is computed between the local data distribution of each client and a reference distribution to measure the degree of non-IID-ness.", "question": "What does PSI measure in the context of federated learning?", "answer": "PSI measures the distributional shift between a client's local data distribution and a reference distribution, quantifying the degree of non-IID-ness in federated learning."},
    {"context": "The silhouette-based procedure is used to determine the optimal number of clusters in Clust-PSI-PFL without requiring a preset number.", "question": "How is the number of clusters determined in Clust-PSI-PFL?", "answer": "The optimal number of clusters is determined using a systematic silhouette-based procedure, which does not require presetting the number of clusters."},
    {"context": "K-means++ clustering is applied on PSI features to group clients with similar data distributions for personalized federated learning.", "question": "What clustering algorithm is used in Clust-PSI-PFL?", "answer": "K-means++ clustering is applied on PSI features to group clients with similar data distributions into clusters for personalized federated learning."},
    {"context": "Clust-PSI-PFL was evaluated on six datasets: ACSIncome, Serengeti, FMNIST, CIFAR10, Sent140, and Amazon reviews.", "question": "What datasets were used to evaluate Clust-PSI-PFL?", "answer": "Clust-PSI-PFL was evaluated on six datasets: ACSIncome, Serengeti, FMNIST, CIFAR10, Sent140, and Amazon reviews."},
    {"context": "The proposed method achieves up to 18% relative gains over strong baselines such as CFL and FedSoft across six datasets.", "question": "What are the main results of Clust-PSI-PFL?", "answer": "Clust-PSI-PFL achieves up to 18% relative performance gains over strong baselines including CFL and FedSoft across six benchmark datasets under various non-IID settings."},
    {"context": "CFL recursively clusters clients via cosine similarity of local updates when training stalls, and is model-agnostic and FedAvg-compatible.", "question": "What is CFL and how does it work?", "answer": "CFL (Clustered Federated Learning) is a baseline method that recursively clusters clients using cosine similarity of their local model updates when training stalls. It is model-agnostic and compatible with FedAvg."},
    {"context": "FedSoft is a baseline federated learning method that uses soft cluster assignments for clients with non-IID data distributions.", "question": "What is FedSoft?", "answer": "FedSoft is a federated learning baseline that uses soft cluster assignments to handle non-IID data distributions across clients."},
    {"context": "Personalized Federated Learning (PFL) addresses non-IID challenges by training personalized models for each client or group of clients.", "question": "What is Personalized Federated Learning?", "answer": "Personalized Federated Learning (PFL) addresses non-IID data challenges by training personalized models tailored to individual clients or groups of clients rather than a single global model."},
    {"context": "The weighted PSI metric (WPSIL) is more informative than other non-IID metrics such as Hellinger, Jensen-Shannon, and Earth Mover's distance.", "question": "How does WPSIL compare to other non-IID metrics?", "answer": "The weighted PSI metric (WPSIL) is more informative than other non-IID metrics including Hellinger distance, Jensen-Shannon divergence, and Earth Mover's distance."},
    {"context": "The research was funded by the MUR National Recovery and Resilience Plan under the project SERICS (PE00000014) as part of the European Union's NextGenerationEU program.", "question": "What funded the Clust-PSI-PFL research?", "answer": "The research was funded by the MUR National Recovery and Resilience Plan under the SERICS project (PE00000014), part of the European Union's NextGenerationEU program."},
    {"context": "FedAvg is the baseline federated learning algorithm that averages client model weights proportionally to their local dataset sizes.", "question": "What is FedAvg?", "answer": "FedAvg is the foundational federated learning algorithm that aggregates client models by computing a weighted average of their parameters proportional to each client's local dataset size."},
    {"context": "Label skew occurs when different clients hold data with different class label distributions, a common form of non-IID data in federated learning.", "question": "What is label skew in federated learning?", "answer": "Label skew is a form of non-IID data distribution where different federated clients hold training data with different class label distributions, causing biased global model updates."},
    {"context": "Attribute skew and quantity skew are additional forms of non-IID data beyond label skew that affect federated learning performance.", "question": "What forms of data skew exist beyond label skew in federated learning?", "answer": "Beyond label skew, federated learning faces attribute skew (different feature distributions) and quantity skew (different amounts of data per client), both of which affect model performance."},
    {"context": "Differential Privacy (DP) and Secure Multi-Party Computation (MPC) are formal privacy mechanisms that can be integrated into federated learning pipelines.", "question": "What privacy mechanisms can strengthen federated learning?", "answer": "Differential Privacy (DP) and Secure Multi-Party Computation (MPC) are formal privacy mechanisms that can be integrated into federated learning pipelines to strengthen client confidentiality."},
    {"context": "PSI is a metric originally used in credit scoring to monitor distributional changes and is repurposed in Clust-PSI-PFL for measuring non-IID-ness.", "question": "What is the original application of PSI before federated learning?", "answer": "PSI was originally used in credit scoring to monitor distributional changes in data over time. Clust-PSI-PFL repurposes it to measure the degree of non-IID-ness between federated clients."},
    {"context": "The global accuracy and fairness metrics AD and SDAD are used to evaluate Clust-PSI-PFL against baseline methods.", "question": "What metrics are used to evaluate Clust-PSI-PFL?", "answer": "Clust-PSI-PFL is evaluated using global accuracy and fairness metrics including AD (accuracy disparity) and SDAD (standard deviation of accuracy disparity) across different datasets and non-IID settings."},
    {"context": "Communication overhead is reduced by 40% compared to gradient-based clustering methods by using PSI as a lightweight client clustering metric.", "question": "How does Clust-PSI-PFL reduce communication overhead?", "answer": "Clust-PSI-PFL reduces communication overhead by 40% compared to gradient-based clustering methods by using PSI as a lightweight, computation-efficient client clustering metric that does not require gradient sharing."},
]

os.makedirs('data', exist_ok=True)
with open('data/fl_qa_pairs.json', 'w') as f:
    json.dump(qa_pairs, f, indent=2)

print(f"✅ {len(qa_pairs)} QA pairs ready")

✅ 20 QA pairs ready


# Cell 15 — Start QLoRA fine-tuning:

In [14]:
from src.finetuning.dataset_prep import prepare_dataset

paths = prepare_dataset(qa_pairs, output_path='data/finetune_dataset.jsonl')
print(f"Train: {paths['train']}")
print(f"Val:   {paths['val']}")

Formatting samples: 100%|██████████| 20/20 [00:00<00:00, 88862.37it/s]
2026-03-21 15:30:47.951 | INFO     | src.finetuning.dataset_prep:prepare_dataset:95 - Train samples: 18 → data/train_finetune_dataset.jsonl
2026-03-21 15:30:47.951 | INFO     | src.finetuning.dataset_prep:prepare_dataset:96 - Val   samples: 2 → data/val_finetune_dataset.jsonl


Train: data/train_finetune_dataset.jsonl
Val:   data/val_finetune_dataset.jsonl


In [15]:
from src.finetuning.train_qlora import QLoRAConfig, load_model_and_tokenizer, apply_lora
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

cfg = QLoRAConfig(
    lora_r=16,
    lora_alpha=32,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
)

# Load dataset
dataset = load_dataset(
    "json",
    data_files={"train": paths['train'], "validation": paths['val']},
)
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])}")

# Load model + apply LoRA (already loaded — reuse if possible)
model, tokenizer = load_model_and_tokenizer(cfg)
model = apply_lora(model, cfg)

# Training args
training_args = TrainingArguments(
    output_dir='models/mistral-qlora-adapter',
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    gradient_checkpointing=True,
    learning_rate=cfg.learning_rate,
    warmup_steps=10,
    lr_scheduler_type=cfg.lr_scheduler_type,
    weight_decay=cfg.weight_decay,
    fp16=cfg.fp16,
    logging_steps=cfg.logging_steps,
    save_strategy=cfg.save_strategy,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_32bit",
)

# Fixed SFTTrainer — use processing_class instead of tokenizer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,       # ← fixed here
    args=training_args,
    dataset_text_field="text",
    max_seq_length=cfg.max_seq_length,
    packing=False,
)

print("Starting QLoRA fine-tuning...")
trainer.train()

# Save adapter
trainer.model.save_pretrained('models/mistral-qlora-adapter')
tokenizer.save_pretrained('models/mistral-qlora-adapter')
print("✅ QLoRA adapter saved to models/mistral-qlora-adapter")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

2026-03-21 15:30:51.713 | INFO     | src.finetuning.train_qlora:load_model_and_tokenizer:74 - Loading mistralai/Mistral-7B-Instruct-v0.2 in 4-bit...


Train: 18 | Val: 2


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'

# Cell 16

In [17]:
from datasets import load_dataset
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer
from src.finetuning.train_qlora import QLoRAConfig, load_model_and_tokenizer, apply_lora

# Dataset already loaded — just retrain
dataset = load_dataset(
    "json",
    data_files={"train": paths['train'], "validation": paths['val']},
)

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding=False,
    )

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Train: {len(tokenized['train'])} | Val: {len(tokenized['validation'])}")

training_args = TrainingArguments(
    output_dir='models/mistral-qlora-adapter',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    weight_decay=0.001,
    bf16=False,       # ← disable bf16
    fp16=False,       # ← disable fp16
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_32bit",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    args=training_args,
    data_collator=data_collator,
)

print("🚀 Starting QLoRA fine-tuning...")
trainer.train()

trainer.model.save_pretrained('models/mistral-qlora-adapter')
tokenizer.save_pretrained('models/mistral-qlora-adapter')
print("✅ Adapter saved to models/mistral-qlora-adapter")

Train: 18 | Val: 2
🚀 Starting QLoRA fine-tuning...


Epoch,Training Loss,Validation Loss
1,No log,3.414057
2,No log,2.647268
3,3.005003,1.847252


✅ Adapter saved to models/mistral-qlora-adapter


Steadily decreasing — the model is learning your domain. Now let's test it against the base model:

In [18]:
import torch, gc
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
from src.pipeline.prompt_templates import RAG_PROMPT

# Free training model from memory
del trainer, model
gc.collect()
torch.cuda.empty_cache()

# Load base Mistral + fine-tuned adapter
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.2',
    quantization_config=bnb_config,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(base_model, 'models/mistral-qlora-adapter')
ft_model = ft_model.merge_and_unload()

ft_pipeline = HuggingFacePipeline(pipeline=pipeline(
    "text-generation", model=ft_model, tokenizer=tokenizer,
    max_new_tokens=512, temperature=0.1, do_sample=True,
))
print("✅ Fine-tuned model loaded!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Fine-tuned model loaded!


In [20]:
import os, gc, torch

# Clear old index
import shutil
if os.path.exists('indexes/faiss'):
    shutil.rmtree('indexes/faiss')

# Rebuild with FL paper only
os.makedirs('data/papers', exist_ok=True)

# Remove attention paper if present
attn_paper = 'data/papers/1706.03762.pdf'
if os.path.exists(attn_paper):
    os.remove(attn_paper)
    print("✅ Removed Attention paper from index")

# Rebuild FAISS + BM25 with only FL paper
from src.ingestion.pdf_loader import PDFLoader
from src.ingestion.chunker import ResearchPaperChunker
from src.retrieval.embeddings import get_embedding_model
from src.retrieval.vector_store import FAISSVectorStore
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_retriever import HybridRetriever

loader  = PDFLoader(source_dir='data/papers')
pages   = loader.load_all()
chunker = ResearchPaperChunker(chunk_size=512, chunk_overlap=64)
chunks  = chunker.chunk(pages)
print(f"✅ Chunks from FL paper only: {len(chunks)}")

embedding_model = get_embedding_model(device='cuda')
faiss_store     = FAISSVectorStore(embedding_model)
faiss_store.build(chunks)
faiss_store.save('indexes/faiss')

bm25   = BM25Retriever.from_documents(chunks, k=5)
hybrid = HybridRetriever(faiss_store=faiss_store, bm25_retriever=bm25, k=5)
print("✅ Index rebuilt with FL paper only")

2026-03-21 15:40:36.939 | INFO     | src.ingestion.pdf_loader:_load_from_directory:69 - Found 1 PDFs in data/papers


✅ Removed Attention paper from index


Loading PDFs: 100%|██████████| 1/1 [00:00<00:00,  4.99it/s]
2026-03-21 15:40:37.142 | INFO     | src.ingestion.pdf_loader:_load_from_directory:73 - Extracted 14 pages total
2026-03-21 15:40:37.148 | INFO     | src.ingestion.chunker:chunk:64 - Chunked 14 pages → 158 chunks (size=512, overlap=64)
2026-03-21 15:40:37.148 | INFO     | src.retrieval.embeddings:get_embedding_model:35 - Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 on cuda


✅ Chunks from FL paper only: 158


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-21 15:40:39.236 | INFO     | src.retrieval.embeddings:get_embedding_model:46 - Embedding model loaded successfully
2026-03-21 15:40:39.237 | INFO     | src.retrieval.vector_store:build:41 - Building FAISS index from 158 chunks...
2026-03-21 15:40:39.720 | INFO     | src.retrieval.vector_store:build:52 - FAISS index built successfully
2026-03-21 15:40:39.722 | INFO     | src.retrieval.vector_store:save:67 - FAISS index saved to indexes/faiss
2026-03-21 15:40:39.731 | INFO     | src.retrieval.bm25_retriever:from_documents:23 - BM25 index built on 158 documents


✅ Index rebuilt with FL paper only


In [21]:
result = ask_ft("What are the main results of this paper?")
print(f"A: {result['result']}")

2026-03-21 15:40:51.794 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: The paper investigates three empirical questions regarding the Clust-PSI-PFL algorithm and its comparison to state-of-the-art baselines. The main results include:
1. Clust-PSI-PFL outperforms competing baselines, including CFL and FedSoft, in terms of accuracy and fairness improvements across various settings.
2. The authors analyze the mean test global accuracy of Clust-PSI-PFL and baselines as a function of the number of clients, Dirichlet parameter α, and Similarity parameter S, reporting up to 18% relative gains for Clust-PSI-PFL.

Source: Context from the provided README.md and the paper itself.


In [22]:
questions = [
    "What is the PSI clustering method in federated learning?",
    "What datasets were used for evaluation?",
    "What are the baseline methods compared in this paper?",
    "What is the main contribution of Clust-PSI-PFL?",
]

for q in questions:
    result = ask_ft(q)
    print(f"Q: {q}")
    print(f"A: {result['result']}")
    print("-" * 60)

2026-03-21 15:45:10.461 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-03-21 15:45:23.168 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the PSI clustering method in federated learning?
A: The PSI clustering method in federated learning is a mechanism for handling non-IID data by partitioning clients into clusters based on their Population Stability Index (PSI) signatures. This method groups clients with similar PSI profiles, which quantifies distributional divergence, to form more homogeneous cohorts and reduce cross-distribution training. (Source: pages 1-5 of the paper "PSI-guided Clustering for Federated Learning under Label Skew" by the authors listed in the context.)
------------------------------------------------------------


2026-03-21 15:45:30.578 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What datasets were used for evaluation?
A: The evaluation was conducted on multiple datasets: ACSIncome [32], Serengeti [33], FMNIST [34], CIFAR10 [35], Sent140 [36], and Amazon reviews [37]. (Table I)
------------------------------------------------------------


2026-03-21 15:45:42.808 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are the baseline methods compared in this paper?
A: The baseline methods compared in this paper include those using FedAvg, FedProx, and FedAdag algorithms for federated learning (FL). Specifically, the authors evaluate the performance of Clust-PSI-PFL, a clustering-based method, against these baselines (FedAvg, FedProx, and FedAdag). The comparison is based on global and local performance metrics, including accuracy, fairness, and variability in client outcomes. (Source: Context, pages 3-5)
------------------------------------------------------------
Q: What is the main contribution of Clust-PSI-PFL?
A: The main contribution of Clust-PSI-PFL is a novel method for federated learning that addresses label skew, a common issue in decentralized data distributions. It utilizes a client-side probability mass function (pmf) and a server-side pmf to estimate class probabilities and identify misclassified samples, respectively. The method also provides per-class terms to identify the cl